# Day 11: Test day
The test week (week 11) is opened **once**, with the system frozen on Day 10. This notebook prepares everything (Steps 1-4) **without looking at any test answers**; Steps 5-9 run each model once and report.

**Checks:** each check prints what I got next to what I expected (no asserts).

## Names used in this notebook
| Name | What it is |
|---|---|
| `train`, `val`, `test` | weeks 0-9, week 10, week 11 |
| `history_w9`, `history_w10` | the data the system may use: weeks 0-9 (for validation), weeks 0-10 (for test) |
| `repurchase_lists(history)` | each customer's own purchases, newest first, at most 10 |
| `last_week_top(history, week, k)` | the k best sellers of one week |
| `age_bucket_top10(history, week)` | the top 10 of that week inside each age bucket |
| `final_system_recommend(...)` | the frozen Day 10 system: repurchase + ALS for warm, age-bucket popularity for cold |
| `tt_customer_map_w9` | the **two-tower** (and old ALS) customer mapping, weeks 0-9 (Day 5) |
| `als_customer_map_w10`, `als_article_map_w10` | the **new ALS** mappings, weeks 0-10 (Step 4). Never mix the two! |
| `manifest` | SHA-256 fingerprints of every frozen file |

## Step 1: The test-day plan (written BEFORE anything is opened)

**Freeze the recipe, refresh the data.** In production the system recomputes its lists every week from the latest data. What was frozen on Day 10 is the **recipe** (rules, order, settings), not the numbers it produced. So for the test week, the system uses data **up to the end of week 10**, exactly as it would on that Monday morning (Option B from Day 4 Step 8).

**1. Refreshed to weeks 0-10:**
- repurchase lists (now include week 10 purchases)
- last-week popularity (now **week 10**)
- age-bucket popularity (now **week 10**)
- ALS, **retrained on weeks 0-10 with the frozen settings read from `models/als/settings.json`** (not retyped)

**2. Frozen, not retrained:** both two-tower models (retraining takes hours). They keep their weeks 0-9 mappings, so a customer first seen in week 10 maps to row 0. That is **slightly unfair to the network**; it's acceptable because the network is not the served system, and the comparison is reported with this caveat.

**3. Compared, each on the same 68,984 test customers:** the final system; global and last-week popularity; repurchase + popularity fill; ALS + popularity fallback; both two-tower models.

**4. The rules:** every model runs **once**; nothing is changed after seeing a number; if test results are below validation, I report them, I don't tune.

**5. Predictions (before opening anything):**
- Final system test recall: **slightly below** its validation 0.0495 (about 0.044-0.049). New articles keep arriving every week and can't be recommended, and every list is one week further from the data the ALS settings were chosen on.
- The gap to popularity **holds** (still more than 2x): it comes from repurchase, which is a stable habit, not a fit to one particular week.

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'      # before numpy: implicit runs its own threads (Day 7)

import hashlib
import json
import math
from datetime import date

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

PROJECT = '..'
FEATURES_FOLDER = PROJECT + '/data/processed/features'
MODELS_FOLDER = PROJECT + '/models'

c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Recommendation System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Checksums of every frozen file
A SHA-256 checksum is a fingerprint of a file: change one byte and the fingerprint changes completely. Recording them **now**, before the test is opened, proves later that nothing was edited after seeing test numbers. ALS is retrained in Step 4, so here only its **settings** are fingerprinted; the new vectors are added after Step 4.

In [2]:
def sha256_of(path):
    # The file's fingerprint: 64 hex characters
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

frozen_files = [
    'models/als/settings.json',
    'models/two_tower/settings.json',
    'models/two_tower/user_tower.keras',
    'models/two_tower/item_tower.keras',
    'models/two_tower_iddrop/settings.json',
    'models/two_tower_iddrop/user_tower.keras',
    'models/two_tower_iddrop/item_tower.keras',
    'data/processed/features/settings.json',
    'data/processed/features/price_scaling.json',
    'data/processed/features/article_median_price.parquet',
    'data/processed/features/type_median_price.parquet',
    'data/processed/train.parquet',
    'data/processed/val.parquet',
    'data/processed/test.parquet',
]
for name in sorted(os.listdir(FEATURES_FOLDER)):
    if name.startswith('vocab_'):
        frozen_files.append('data/processed/features/' + name)

manifest = {'created': str(date.today()), 'files': {}}
for path in frozen_files:
    manifest['files'][path] = sha256_of(PROJECT + '/' + path)

MANIFEST_PATH = PROJECT + '/test_day_manifest.json'
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print('Files fingerprinted:', len(manifest['files']))
for path in manifest['files']:
    print(' ', manifest['files'][path][:12] + '...', path)

Files fingerprinted: 29
  ac5a26074165... models/als/settings.json
  46aa5c07cafb... models/two_tower/settings.json
  60c63153d767... models/two_tower/user_tower.keras
  7c31b4dedc69... models/two_tower/item_tower.keras
  ea1c9cc81979... models/two_tower_iddrop/settings.json
  d5b69cefc944... models/two_tower_iddrop/user_tower.keras
  050369652ca9... models/two_tower_iddrop/item_tower.keras
  9f376929fe0b... data/processed/features/settings.json
  fd0811d396e6... data/processed/features/price_scaling.json
  46f1d17fdd7b... data/processed/features/article_median_price.parquet
  03988e487fa3... data/processed/features/type_median_price.parquet
  08708be9b015... data/processed/train.parquet
  5de325145924... data/processed/val.parquet
  3b76fd468a6c... data/processed/test.parquet
  15cd07df623b... data/processed/features/vocab_age_bucket.parquet
  fca80f7c4c14... data/processed/features/vocab_article.parquet
  fbe53659461f... data/processed/features/vocab_club_member_status.parquet
  50b8

## Step 3: The test answer key (counts only)
Built exactly like validation's (Day 3 Step 5), with `is_cold` against **weeks 0-10** (train + validation customers). **Only counts are printed**: building the answer key is setting up the exam, not reading the answers.

In [3]:
train = pd.read_parquet(PROJECT + '/data/processed/train.parquet')
val = pd.read_parquet(PROJECT + '/data/processed/val.parquet')
test = pd.read_parquet(PROJECT + '/data/processed/test.parquet')

history_w9 = train                                          # what the system may use for validation
history_w10 = pd.concat([train, val], ignore_index=True)    # what it may use for test
print('Weeks 0-9 rows: ', len(history_w9), '| expected 2,638,103')
print('Weeks 0-10 rows:', len(history_w10), '| expected 2,869,083 (2,638,103 + 230,980)')

test_key = test.groupby('customer_join_idx')['article_id'].unique().reset_index()
test_key['is_cold'] = ~test_key['customer_join_idx'].isin(history_w10['customer_join_idx'].unique())
print('Test customers:', len(test_key), '| expected 68,984 (Day 4)')
print('Cold test customers:', test_key['is_cold'].sum(), '| expected 21,610 (Day 4)')

Weeks 0-9 rows:  2638103 | expected 2,638,103
Weeks 0-10 rows: 2869083 | expected 2,869,083 (2,638,103 + 230,980)
Test customers: 68984 | expected 68,984 (Day 4)
Cold test customers: 21610 | expected 21,610 (Day 4)


## Step 4: Refresh the inputs, with one parity check
No separate "test version" code: every input is a **function of the history it may use**. Called with weeks 0-9 and week 9, the functions must reproduce the Day 10 validation result **exactly (0.0495)**. That proves the refresh code *is* the validation code. Only then are they called with weeks 0-10 and week 10.

### 4a. Features and helpers (copied unchanged from 07, which was checked against 05 and 06)

In [4]:
customers = pd.read_parquet(PROJECT + '/data/RAW/customers.parquet')
customers_idx = pd.read_parquet(PROJECT + '/data/processed/customer_join_idx.parquet')
customers_with_idx = customers.merge(customers_idx, on='customer_id', how='inner', validate='one_to_one')


def load_vocab(name):
    table = pd.read_parquet(FEATURES_FOLDER + '/vocab_' + name + '.parquet')
    return dict(zip(table['value'], table['index']))

tt_customer_map_w9 = load_vocab('customer')     # the Day 5 mapping: two-tower and the OLD ALS use it
tt_article_map_w9 = load_vocab('article')
age_bucket_vocab = load_vocab('age_bucket')
club_member_status_vocab = load_vocab('club_member_status')
fashion_news_frequency_vocab = load_vocab('fashion_news_frequency')
with open(FEATURES_FOLDER + '/settings.json') as f:
    age_edges = json.load(f)['age_edges']


def build_customer_features(customer_join_ids):
    # same function as 06 and 07 (copied unchanged)
    table = customers_with_idx[customers_with_idx['customer_join_idx'].isin(customer_join_ids)].copy()
    table['FN'] = table['FN'].fillna(0)
    table['Active'] = table['Active'].fillna(0)
    table['fashion_news_frequency'] = table['fashion_news_frequency'].replace('None', 'NONE')
    table['fashion_news_frequency'] = table['fashion_news_frequency'].map(fashion_news_frequency_vocab).fillna(0)
    table['club_member_status'] = table['club_member_status'].map(club_member_status_vocab).fillna(0)
    table['age'] = pd.cut(table['age'], bins=age_edges, right=False, labels=age_edges[:-1]).astype('float')
    table['age'] = table['age'].map(age_bucket_vocab).fillna(0)
    table['customer_idx'] = table['customer_join_idx'].map(tt_customer_map_w9).fillna(0)
    table = table.rename(columns={
        'age': 'age_bucket_idx',
        'club_member_status': 'club_member_status_idx',
        'fashion_news_frequency': 'fashion_news_frequency_idx',
    })
    return table[['customer_join_idx', 'customer_idx', 'age_bucket_idx', 'club_member_status_idx',
                  'fashion_news_frequency_idx', 'FN', 'Active']]


def fill_up(my_list, fallback, k=10):
    filled = list(my_list)
    for article in fallback:
        if len(filled) == k:
            break
        if article not in filled:
            filled.append(article)
    return filled


def recall_at_k(recommended, bought, k):
    hits = 0
    for item in recommended[:k]:
        if item in bought:
            hits = hits + 1
    return hits / len(bought)


def ndcg_at_k(recommended, bought, k):
    dcg = 0
    position = 0
    for item in recommended[:k]:
        position = position + 1
        if item in bought:
            dcg = dcg + 1 / math.log2(position + 1)
    ideal = 0
    for position in range(1, min(len(bought), k) + 1):
        ideal = ideal + 1 / math.log2(position + 1)
    return dcg / ideal


def score(recommendations, bought_lists):
    recall_list = []
    ndcg_list = []
    for recs, bought in zip(recommendations, bought_lists):
        recall_list.append(recall_at_k(list(recs), set(bought), 10))
        ndcg_list.append(ndcg_at_k(list(recs), set(bought), 10))
    return np.array(recall_list), np.array(ndcg_list)

### 4b. The three refreshable inputs, as functions of the history

In [5]:
def repurchase_lists(history):
    # Each customer's own purchases: newest first, each article once, at most 10
    newest_first = history.sort_values('t_dat', ascending=False)
    newest_first = newest_first.drop_duplicates(subset=['customer_join_idx', 'article_id'], keep='first')
    newest_first = newest_first.groupby('customer_join_idx').head(10)
    return newest_first.groupby('customer_join_idx')['article_id'].apply(list).to_dict()


def last_week_top(history, week, k):
    # The k best sellers of one week, most bought first
    return history[history['week'] == week]['article_id'].value_counts().head(k).index.to_list()


def age_bucket_top10(history, week):
    # The top 10 of that week inside each age bucket (by the BUYER's age). Bucket 0 (age unknown) is left out
    week_rows = history[history['week'] == week][['customer_join_idx', 'article_id']]
    buyers = build_customer_features(week_rows['customer_join_idx'].unique())[['customer_join_idx', 'age_bucket_idx']]
    week_rows = week_rows.merge(buyers, on='customer_join_idx', how='left', validate='many_to_one')
    lists = {}
    for bucket in sorted(week_rows['age_bucket_idx'].unique()):
        if bucket == 0:
            continue
        lists[bucket] = week_rows[week_rows['age_bucket_idx'] == bucket]['article_id'].value_counts().head(10).index.to_list()
    return lists

### 4c. ALS recommend, and the whole frozen system as one function
The ALS parts (vectors and mappings) are **arguments**, so the same function runs with the old ALS (weeks 0-9, for the parity check) and the new one (weeks 0-10, for test).

In [6]:
def als_recommend(customer_ids, k, user_factors, item_factors, customer_map, column_to_article):
    # ALS top-k for customers that exist in customer_map (dot product = filter off, like Day 8)
    rows = np.array([customer_map[c] for c in customer_ids])
    top_columns = []
    for start in range(0, len(rows), 2000):
        chunk_scores = user_factors[rows[start:start + 2000]] @ item_factors.T
        top = np.argpartition(-chunk_scores, k, axis=1)[:, :k]                       # the k best, unordered
        order = np.take_along_axis(chunk_scores, top, axis=1).argsort(axis=1)[:, ::-1]  # sort those k, best first
        top_columns.append(np.take_along_axis(top, order, axis=1))
    return column_to_article[np.vstack(top_columns)]


def column_to_article_array(article_map):
    # ALS column number -> article ID (column 0 = unknown)
    array = np.zeros(len(article_map) + 1, dtype='int64')
    for article, column in article_map.items():
        array[column] = article
    return array


def final_system_recommend(customer_ids, history, week, als_parts):
    # The FROZEN Day 10 system. Warm (in history): own purchases, then ALS fills (30 candidates).
    # Cold: last-week popularity in their age bucket, filled up from the global list. Returns one list per customer
    own_lists = repurchase_lists(history)
    global_top10 = last_week_top(history, week, 10)
    bucket_lists = age_bucket_top10(history, week)

    known = set(history['customer_join_idx'].unique())
    warm_ids = [c for c in customer_ids if c in known]
    cold_ids = [c for c in customer_ids if c not in known]

    recommendations = {}
    als_candidates = als_recommend(warm_ids, 30, *als_parts)
    for customer, candidates in zip(warm_ids, als_candidates):
        recommendations[customer] = fill_up(own_lists[customer], candidates)

    cold_buckets = build_customer_features(cold_ids).set_index('customer_join_idx')['age_bucket_idx']
    for customer in cold_ids:
        recommendations[customer] = fill_up(bucket_lists.get(cold_buckets[customer], []), global_top10)

    return [recommendations[c] for c in customer_ids]

### 4d. Parity: weeks 0-9 + week 9 + the old ALS must reproduce the Day 10 validation result exactly

In [7]:
val_key = val.groupby('customer_join_idx')['article_id'].unique().reset_index()
val_key['is_cold'] = ~val_key['customer_join_idx'].isin(history_w9['customer_join_idx'].unique())

old_als_parts = (np.load(MODELS_FOLDER + '/als/user_factors.npy'),
                 np.load(MODELS_FOLDER + '/als/item_factors.npy'),
                 tt_customer_map_w9,
                 column_to_article_array(tt_article_map_w9))

val_lists = final_system_recommend(list(val_key['customer_join_idx']), history_w9, 9, old_als_parts)
lengths = [len(recs) for recs in val_lists]
print('List lengths:', min(lengths), 'to', max(lengths), '| expected 10 to 10')

val_recall, val_ndcg = score(val_lists, val_key['article_id'])
is_cold = val_key['is_cold'].to_numpy()
print('Validation all  recall =', round(val_recall.mean(), 4), ' NDCG =', round(val_ndcg.mean(), 4), '| expected 0.0495 / 0.0368')
print('Validation warm recall =', round(val_recall[~is_cold].mean(), 4), ' NDCG =', round(val_ndcg[~is_cold].mean(), 4), '| expected 0.0631 / 0.0478')
print('Validation cold recall =', round(val_recall[is_cold].mean(), 4), ' NDCG =', round(val_ndcg[is_cold].mean(), 4), '| expected 0.0216 / 0.0141')

List lengths: 10 to 10 | expected 10 to 10
Validation all  recall = 0.0495  NDCG = 0.0368 | expected 0.0495 / 0.0368
Validation warm recall = 0.0631  NDCG = 0.0478 | expected 0.0631 / 0.0478
Validation cold recall = 0.0216  NDCG = 0.0141 | expected 0.0216 / 0.0141


### 4e. Refresh with weeks 0-10 (week 10 is now "last week")
Nothing is scored here: these are the inputs the test run will use in Step 5.

In [8]:
own_lists_w10 = repurchase_lists(history_w10)
global_top10_w10 = last_week_top(history_w10, 10, 10)
bucket_lists_w10 = age_bucket_top10(history_w10, 10)

print('Customers with a repurchase list:', len(own_lists_w10), '| expected 467,943 (444,431 + 23,512 new in week 10)')
print('Week 10 global top 10:', len(global_top10_w10), 'articles')
print('Age buckets with a week 10 list:', len(bucket_lists_w10), '| expected 7')

Customers with a repurchase list: 467943 | expected 467,943 (444,431 + 23,512 new in week 10)
Week 10 global top 10: 10 articles
Age buckets with a week 10 list: 7 | expected 7


### 4f. Retrain ALS on weeks 0-10 with the frozen settings
**Predicted mapping sizes (before running):**
- customers = 444,431 + the 23,512 customers first seen in week 10 (Day 4: the cold validation customers) = **467,943**
- articles = 38,603 + the 1,225 cold validation articles (Day 4) = **39,828**

**The trap:** the new ALS uses the **new** mappings; the two-tower models keep the **old** ones. Different names (`als_..._w10` vs `tt_..._w9`), so a lookup never goes to the wrong table (that would give wrong rows with no error).

In [9]:
# New mappings, built like Day 5's build_vocab: sorted values -> 1..N (0 = unknown)
als_customer_map_w10 = {}
for number, customer in enumerate(sorted(history_w10['customer_join_idx'].unique()), start=1):
    als_customer_map_w10[customer] = number
als_article_map_w10 = {}
for number, article in enumerate(sorted(history_w10['article_id'].unique()), start=1):
    als_article_map_w10[article] = number

print('ALS customers (weeks 0-10):', len(als_customer_map_w10), '| predicted 467,943')
print('ALS articles  (weeks 0-10):', len(als_article_map_w10), '| predicted 39,828')
print('Old two-tower mapping still:', len(tt_customer_map_w9), '| expected 444,431 (unchanged)')

ALS customers (weeks 0-10): 467943 | predicted 467,943
ALS articles  (weeks 0-10): 39828 | predicted 39,828
Old two-tower mapping still: 444431 | expected 444,431 (unchanged)


In [10]:
# The matrix, exactly like Day 7 Step 3-4: times bought per (customer, article)
pair_counts_w10 = history_w10.groupby(['customer_join_idx', 'article_id']).size().reset_index(name='times_bought')
pair_counts_w10['row'] = pair_counts_w10['customer_join_idx'].map(als_customer_map_w10)
pair_counts_w10['col'] = pair_counts_w10['article_id'].map(als_article_map_w10)

user_items_w10 = csr_matrix(
    (pair_counts_w10['times_bought'].astype('float32'), (pair_counts_w10['row'], pair_counts_w10['col'])),
    shape=(len(als_customer_map_w10) + 1, len(als_article_map_w10) + 1),
)
print('Shape:       ', user_items_w10.shape, '| expected', (len(als_customer_map_w10) + 1, len(als_article_map_w10) + 1))
print('Filled cells:', user_items_w10.nnz, '| expected', len(pair_counts_w10), '(unique pairs)')
print('Sum of cells:', int(user_items_w10.sum()), '| expected', len(history_w10), '(every row adds 1 to one cell)')
print('Missing rows/cols:', pair_counts_w10['row'].isna().sum(), pair_counts_w10['col'].isna().sum(), '| expected 0 0')

Shape:        (467944, 39829) | expected (467944, 39829)
Filled cells: 2784442 | expected 2784442 (unique pairs)
Sum of cells: 2869083 | expected 2869083 (every row adds 1 to one cell)
Missing rows/cols: 0 0 | expected 0 0


In [11]:
# Train with the FROZEN settings, read from the file (never retyped)
with open(MODELS_FOLDER + '/als/settings.json') as f:
    als_settings = json.load(f)
print('Frozen ALS settings:', als_settings)

als_w10 = AlternatingLeastSquares(
    factors=als_settings['factors'],
    iterations=als_settings['iterations'],
    regularization=als_settings['regularization'],
    alpha=als_settings['alpha'],
    random_state=als_settings['random_state'],
    use_gpu=False,
)
als_w10.fit(user_items_w10)
print('user_factors:', als_w10.user_factors.shape, '| item_factors:', als_w10.item_factors.shape)

Frozen ALS settings: {'factors': 64, 'iterations': 15, 'regularization': 0.05, 'alpha': 40, 'random_state': 42, 'filter_already_liked_items': False, 'implicit_version': '0.7.3'}


100%|██████████| 15/15 [00:35<00:00,  2.38s/it]

user_factors: (467944, 64) | item_factors: (39829, 64)


In [12]:
# Save the new ALS (vectors + mappings + settings) to a NEW folder, then add their fingerprints to the manifest
ALS_W10_FOLDER = MODELS_FOLDER + '/als_w0_10'
os.makedirs(ALS_W10_FOLDER, exist_ok=True)
np.save(ALS_W10_FOLDER + '/user_factors.npy', als_w10.user_factors)
np.save(ALS_W10_FOLDER + '/item_factors.npy', als_w10.item_factors)
pd.DataFrame({'value': list(als_customer_map_w10.keys()), 'index': list(als_customer_map_w10.values())}).to_parquet(ALS_W10_FOLDER + '/customer_map.parquet')
pd.DataFrame({'value': list(als_article_map_w10.keys()), 'index': list(als_article_map_w10.values())}).to_parquet(ALS_W10_FOLDER + '/article_map.parquet')

als_w10_settings = dict(als_settings)
als_w10_settings['trained_on'] = 'weeks 0-10 (train + validation)'
als_w10_settings['customers'] = len(als_customer_map_w10)
als_w10_settings['articles'] = len(als_article_map_w10)
with open(ALS_W10_FOLDER + '/settings.json', 'w') as f:
    json.dump(als_w10_settings, f, indent=2)

for name in ['user_factors.npy', 'item_factors.npy', 'customer_map.parquet', 'article_map.parquet', 'settings.json']:
    path = 'models/als_w0_10/' + name
    manifest['files'][path] = sha256_of(PROJECT + '/' + path)
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Manifest now has', len(manifest['files']), 'files')

new_als_parts = (als_w10.user_factors, als_w10.item_factors, als_customer_map_w10,
                 column_to_article_array(als_article_map_w10))
print('Ready for Step 5: new_als_parts built from the weeks 0-10 ALS')

Manifest now has 34 files
Ready for Step 5: new_als_parts built from the weeks 0-10 ALS


## Step 5: Run every model once, and save the lists
**Generate first, score later.** All lists are generated and saved before any test score is computed, so nothing about one model's score can influence how the next one is run, and the table can be recomputed later from the saved files.

Only the models from the Step 1 plan:
1. **Final system** (frozen Day 10 recipe, refreshed data)
2. **Global top 10** (all weeks of history)
3. **Last-week top 10**
4. **Repurchase + last-week fill**
5. **ALS + last-week fallback** (warm: ALS top 10, cold: last-week top 10)
6. **Two-tower Day 9** (frozen at week 9)
7. **Two-tower Day 10, ID dropout** (frozen at week 9)

### 5a. Two-tower setup (copied unchanged from 07, where it was checked against 06)

In [13]:
import tensorflow as tf
from tensorflow import keras

articles = pd.read_parquet(PROJECT + '/data/RAW/articles.parquet')
with open(FEATURES_FOLDER + '/price_scaling.json') as f:
    price_scaling = json.load(f)

# The two-tower models only know the weeks 0-9 articles (frozen at week 9)
tt_article_table = articles[articles['article_id'].isin(train['article_id'].unique())].copy()
tt_article_table['article_idx'] = tt_article_table['article_id'].map(tt_article_map_w9)
article_columns = ['product_code', 'product_type_no', 'product_group_name', 'graphical_appearance_name',
                   'colour_group_name', 'perceived_colour_master_name', 'department_no',
                   'index_group_name', 'section_name', 'garment_group_name']
for name in article_columns:
    tt_article_table[name + '_idx'] = tt_article_table[name].map(load_vocab(name)).fillna(0)
tt_article_table = tt_article_table.merge(pd.read_parquet(FEATURES_FOLDER + '/article_median_price.parquet'),
                                          on='article_id', how='left', validate='one_to_one')
tt_article_table['price_scaled'] = ((np.log(tt_article_table['median_price']) - price_scaling['price_log_mean'])
                                    / price_scaling['price_log_std'])
tt_row_to_article_id = tt_article_table['article_id'].to_numpy()

day9_user = keras.models.load_model(MODELS_FOLDER + '/two_tower/user_tower.keras')
day9_item = keras.models.load_model(MODELS_FOLDER + '/two_tower/item_tower.keras')
day10_user = keras.models.load_model(MODELS_FOLDER + '/two_tower_iddrop/user_tower.keras')
day10_item = keras.models.load_model(MODELS_FOLDER + '/two_tower_iddrop/item_tower.keras')


def tower_inputs(tower, table):
    inputs = {}
    for name in tower.input:
        inputs[name] = table[name].to_numpy().astype(tower.input[name].dtype)
    return inputs


def two_tower_recommend(user_tower, item_tower, customer_ids, k):
    features = pd.DataFrame({'customer_join_idx': customer_ids}).merge(
        build_customer_features(customer_ids), on='customer_join_idx', how='left', validate='one_to_one')
    user_vectors = user_tower.predict(tower_inputs(user_tower, features), batch_size=4096, verbose=0)
    article_vectors = item_tower.predict(tower_inputs(item_tower, tt_article_table), batch_size=4096, verbose=0)
    top_rows = []
    for start in range(0, len(user_vectors), 2000):
        top_rows.append(tf.math.top_k(user_vectors[start:start + 2000] @ article_vectors.T, k=k).indices.numpy())
    return tt_row_to_article_id[np.vstack(top_rows)]

print('Two-tower articles:', len(tt_article_table), '| expected 38,603 (weeks 0-9 only)')

Two-tower articles: 38603 | expected 38,603 (weeks 0-9 only)


### 5b. One function per model
Each takes the customers, the history it may use, "last week", and the ALS parts, and returns one list of 10 per customer **in the same order**.

In [14]:
def recommend_global(customer_ids, history, week, als_parts):
    top10 = history['article_id'].value_counts().head(10).index.to_list()       # all weeks of history
    return [top10] * len(customer_ids)


def recommend_last_week(customer_ids, history, week, als_parts):
    return [last_week_top(history, week, 10)] * len(customer_ids)


def recommend_repurchase(customer_ids, history, week, als_parts):
    own_lists = repurchase_lists(history)
    top30 = last_week_top(history, week, 30)
    lists = []
    for customer in customer_ids:
        if customer in own_lists:
            lists.append(fill_up(own_lists[customer], top30))
        else:
            lists.append(top30[:10])                         # cold: last-week top 10
    return lists


def recommend_als_fallback(customer_ids, history, week, als_parts):
    known = set(history['customer_join_idx'].unique())
    warm_ids = [c for c in customer_ids if c in known]
    als_lists = dict(zip(warm_ids, als_recommend(warm_ids, 10, *als_parts)))
    top10 = last_week_top(history, week, 10)
    return [list(als_lists[c]) if c in als_lists else top10 for c in customer_ids]


def recommend_final(customer_ids, history, week, als_parts):
    return final_system_recommend(customer_ids, history, week, als_parts)


def recommend_two_tower_day9(customer_ids, history, week, als_parts):
    return [list(row) for row in two_tower_recommend(day9_user, day9_item, customer_ids, 10)]


def recommend_two_tower_day10(customer_ids, history, week, als_parts):
    return [list(row) for row in two_tower_recommend(day10_user, day10_item, customer_ids, 10)]


models = {
    'Final system': recommend_final,
    'Global top 10': recommend_global,
    'Last-week top 10': recommend_last_week,
    'Repurchase + last-week fill': recommend_repurchase,
    'ALS + last-week fallback': recommend_als_fallback,
    'Two-tower Day 9': recommend_two_tower_day9,
    'Two-tower Day 10 (ID dropout)': recommend_two_tower_day10,
}

### 5c. Parity on validation BEFORE touching the test week
Every model, run with weeks 0-9 / week 9 / the old ALS, must reproduce its known validation number. A wrong generator is caught here, not after the test is opened. These validation numbers also go into the Step 6 table (measured, not typed).

In [15]:
expected_val = {'Final system': 0.0495, 'Global top 10': 0.0090, 'Last-week top 10': 0.0184,
                'Repurchase + last-week fill': 0.0457, 'ALS + last-week fallback': 0.0327,
                'Two-tower Day 9': 0.0218, 'Two-tower Day 10 (ID dropout)': 0.0251}

val_ids = list(val_key['customer_join_idx'])
val_recall_all = {}
for name in models:
    lists = models[name](val_ids, history_w9, 9, old_als_parts)
    recall, ndcg = score(lists, val_key['article_id'])
    val_recall_all[name] = recall.mean()
    print(name.ljust(30), 'validation recall@10 =', round(recall.mean(), 4), '| expected', expected_val[name])

Final system                   validation recall@10 = 0.0495 | expected 0.0495
Global top 10                  validation recall@10 = 0.009 | expected 0.009
Last-week top 10               validation recall@10 = 0.0184 | expected 0.0184
Repurchase + last-week fill    validation recall@10 = 0.0457 | expected 0.0457
ALS + last-week fallback       validation recall@10 = 0.0327 | expected 0.0327
Two-tower Day 9                validation recall@10 = 0.0218 | expected 0.0218
Two-tower Day 10 (ID dropout)  validation recall@10 = 0.0251 | expected 0.0251


### 5d. Generate the test lists ONCE, check them, save them (no scores here)
Same customer order for every model (`test_key`). Checks: 68,984 lists, each exactly 10 articles, no duplicates inside a list.

In [16]:
# Safety gate: if ANY generator failed its validation parity, stop here, before the test week is touched
mismatched = [name for name in models if round(val_recall_all[name], 4) != expected_val[name]]
if mismatched:
    raise RuntimeError('Validation parity failed for: ' + str(mismatched) + ' -> fix before generating test lists')
print('All 7 generators reproduced their validation numbers -> generating test lists')

LISTS_FOLDER = PROJECT + '/results/test_lists'
os.makedirs(LISTS_FOLDER, exist_ok=True)
test_ids = list(test_key['customer_join_idx'])

for name in models:
    lists = models[name](test_ids, history_w10, 10, new_als_parts)

    sizes = [len(recs) for recs in lists]
    distinct_sizes = [len(set(recs)) for recs in lists]
    print(name.ljust(30), '| lists:', len(lists), '| sizes', min(sizes), 'to', max(sizes),
          '| no duplicates:', min(distinct_sizes) == 10, '| expected 68984, 10 to 10, True')

    table = pd.DataFrame(np.array(lists, dtype='int64'), columns=['rec_' + str(i) for i in range(1, 11)])
    table.insert(0, 'customer_join_idx', test_ids)
    file_name = name.lower().replace(' ', '_').replace('+', 'plus').replace('(', '').replace(')', '').replace('-', '_')
    table.to_parquet(LISTS_FOLDER + '/' + file_name + '.parquet')

print(sorted(os.listdir(LISTS_FOLDER)))

All 7 generators reproduced their validation numbers -> generating test lists
Final system                   | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
Global top 10                  | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
Last-week top 10               | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
Repurchase + last-week fill    | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
ALS + last-week fallback       | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
Two-tower Day 9                | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
Two-tower Day 10 (ID dropout)  | lists: 68984 | sizes 10 to 10 | no duplicates: True | expected 68984, 10 to 10, True
['als_plus_last_week_fallback.parquet', 'final_system.parquet', 'global_top_10.parquet', 'last_w

## Step 6: The test results table
Lists are loaded back from the saved files and scored with the same metric functions.

**Catalogue coverage** = distinct articles appearing in any of the 68,984 lists ÷ the 39,828 articles sold in weeks 0-10. Recall only rewards hits; coverage shows how much of the shop a system actually uses (unsold stock is lost money).

**Coverage prediction (before computing):** most coverage: the **final system** (every customer's own purchases plus personal ALS suggestions), then repurchase (own purchases), then ALS, then the two-tower models; least: **global and last-week top 10** (10 articles each, 0.03%).

In [17]:
is_cold_test = test_key['is_cold'].to_numpy()
rows = []
for name in models:
    file_name = name.lower().replace(' ', '_').replace('+', 'plus').replace('(', '').replace(')', '').replace('-', '_')
    table = pd.read_parquet(LISTS_FOLDER + '/' + file_name + '.parquet')
    lists = table[['rec_' + str(i) for i in range(1, 11)]].to_numpy()

    recall, ndcg = score(lists, test_key['article_id'])
    coverage = len(np.unique(lists)) / len(als_article_map_w10)
    rows.append({
        'Model': name,
        'Recall all': round(recall.mean(), 4), 'NDCG all': round(ndcg.mean(), 4),
        'Recall warm': round(recall[~is_cold_test].mean(), 4), 'NDCG warm': round(ndcg[~is_cold_test].mean(), 4),
        'Recall cold': round(recall[is_cold_test].mean(), 4), 'NDCG cold': round(ndcg[is_cold_test].mean(), 4),
        'Val recall all': round(val_recall_all[name], 4),
        'Test vs val %': round((recall.mean() - val_recall_all[name]) / val_recall_all[name] * 100),
        'Coverage %': round(coverage * 100, 2),
    })

test_table = pd.DataFrame(rows)
print('Test customers:', len(test_key), '| warm', (~is_cold_test).sum(), '| cold', is_cold_test.sum())
test_table

Test customers: 68984 | warm 47374 | cold 21610


,Model,Recall all,NDCG all,Recall warm,NDCG warm,Recall cold,NDCG cold,Val recall all,Test vs val %,Coverage %
0,Final system,0.0507,0.0370,0.0625,0.0467,0.0250,0.0157,0.0495,2,60.94
1,Global top 10,0.0122,0.0082,0.0112,0.0075,0.0146,0.0098,0.0090,36,0.03
2,Last-week top 10,0.0222,0.0145,0.0221,0.0148,0.0224,0.0136,0.0184,21,0.03
3,Repurchase + last-week fill,0.0495,0.0361,0.0619,0.0463,0.0224,0.0136,0.0457,8,60.94
4,ALS + last-week fallback,0.0343,0.0228,0.0397,0.0270,0.0224,0.0136,0.0327,5,9.39
5,Two-tower Day 9,0.0119,0.0084,0.0163,0.0115,0.0023,0.0016,0.0218,-45,40.61
6,Two-tower Day 10 (ID dropout),0.0167,0.0112,0.0178,0.0120,0.0142,0.0094,0.0251,-33,32.95


In [18]:
# Did the ranking of the models stay the same from validation to test?
ranking = test_table[['Model', 'Val recall all', 'Recall all']].copy()
ranking['Val rank'] = ranking['Val recall all'].rank(ascending=False).astype(int)
ranking['Test rank'] = ranking['Recall all'].rank(ascending=False).astype(int)
ranking.sort_values('Test rank')

,Model,Val recall all,Recall all,Val rank,Test rank
0,Final system,0.0495,0.0507,1,1
3,Repurchase + last-week fill,0.0457,0.0495,2,2
4,ALS + last-week fallback,0.0327,0.0343,3,3
2,Last-week top 10,0.0184,0.0222,6,4
6,Two-tower Day 10 (ID dropout),0.0251,0.0167,4,5
1,Global top 10,0.0090,0.0122,7,6
5,Two-tower Day 9,0.0218,0.0119,5,7


**Written answers (after the single run):**

1. **Did the ranking hold?** The **top three held exactly**: final system 1st, repurchase + fill 2nd, ALS + fallback 3rd, on both validation and test. So the Day 10 decisions (repurchase first, ALS as filler) were sound. The changes are all below them: both two-tower models dropped (Day 10 from 4th to 5th, Day 9 from 5th to 7th) and last-week popularity rose from 6th to 4th. That is consistent with the caveat: the two-tower models are frozen at week 9, so the 23,512 customers first seen in week 10 are row 0 for them, and their article set stops at week 9, while every other model was refreshed with week 10.

2. **The final system: 0.0507, above my predicted range of 0.044-0.049** (+2% vs validation). Every refreshed model scored *higher* on test than on validation (popularity +21-36%, repurchase +8%), so the test week was easier overall, probably because it was quieter (Day 3): fewer purchases per customer, more concentrated on bestsellers. The refreshed system kept its lead: 2.3x last-week popularity on test (2.7x on validation). Cold improved most (0.0250 vs 0.0216): age-bucket lists from week 10 were one week fresher than on validation.

3. **The two-tower models: well behind, and further behind than on validation** (Day 10: 0.0167 test vs 0.0251 validation, −33%; Day 9 −45%). On validation, Day 10 was 36% above last-week popularity; on test it is 25% below it. That gap is mostly the frozen-at-week-9 caveat (staleness and row 0 for week 10's customers), so it says the network was not refreshed, not that it got worse; but even on validation it never beat the final system.

**Coverage:** my prediction held at both ends: final system (60.9%) and repurchase (60.9%) highest, global and last-week top 10 lowest (0.03%, 10 articles). Final and repurchase are identical to one decimal because coverage is dominated by customers' own purchases; the ALS and popularity fillers add articles that already appear in someone else's own list. One surprise: the two-tower models cover far more of the catalogue (33-41%) than ALS (9.4%), so ALS concentrates on fewer, more popular articles while the network spreads out.

## Step 7: Look inside the network's item vectors
The network isn't served, but it learned something. Day 10 item tower, all 38,603 weeks 0-9 articles; vectors are length 1, so a dot product is the cosine.

**Prediction:** same-`product_code` colour variants **will** be nearest neighbours: the item tower gets a 32-number `product_code` embedding, and colour variants share it, plus the same product type, department, section and garment group. Colour is only 2 of the 12 inputs, so it should matter less.

In [19]:
item_vectors = day10_item.predict(tower_inputs(day10_item, tt_article_table), batch_size=4096, verbose=0)
article_row = {article: row for row, article in enumerate(tt_row_to_article_id)}
info = articles.set_index('article_id')[['product_code', 'prod_name', 'product_type_name', 'colour_group_name']]

train_counts = train['article_id'].value_counts()
week9_bucket1 = age_bucket_top10(history_w9, 9)[1.0]
global_w9 = last_week_top(history_w9, 9, 10)

picks = {
    'Top seller': train_counts.index[0],
    'Basic (socks)': train_counts[train_counts.index.isin(info[info['product_type_name'] == 'Socks'].index)].index[0],
    'Dress': train_counts[train_counts.index.isin(info[info['product_type_name'] == 'Dress'].index)].index[0],
    'Ages 16-19 favourite': [a for a in week9_bucket1 if a not in global_w9][0],
    'Rarely bought (once)': train_counts[train_counts == 1].index[0],
}

for label, article in picks.items():
    scores = item_vectors @ item_vectors[article_row[article]]
    order = np.argsort(scores)[::-1]
    print('=' * 100)
    print(label + ':', article, '|', info.loc[article, 'prod_name'], '|', info.loc[article, 'product_type_name'],
          '|', info.loc[article, 'colour_group_name'], '| bought', train_counts[article], 'times')
    shown = 0
    for row in order:
        neighbour = tt_row_to_article_id[row]
        if neighbour == article:
            continue
        same_code = info.loc[neighbour, 'product_code'] == info.loc[article, 'product_code']
        print('   ', round(float(scores[row]), 3), '|', str(info.loc[neighbour, 'prod_name'])[:28].ljust(28), '|',
              str(info.loc[neighbour, 'product_type_name']).ljust(16), '|', str(info.loc[neighbour, 'colour_group_name']).ljust(16),
              '| same product_code:', same_code)
        shown = shown + 1
        if shown == 5:
            break

Top seller: 706016001 | Jade HW Skinny Denim TRS | Trousers | Black | bought 3853 times
    0.936 | Jade HW Skinny Denim TRS     | Trousers         | Dark Blue        | same product_code: True
    0.866 | Victorville HW Pull-on TRS   | Trousers         | Black            | same product_code: False
    0.864 | Jade HW Skinny Denim TRS     | Trousers         | Light Blue       | same product_code: True
    0.848 | Perrie Slim Mom Denim TRS    | Trousers         | Blue             | same product_code: False
    0.837 | Jade HW Skinny Denim TRS     | Trousers         | Blue             | same product_code: True
Basic (socks): 372860002 | 7p Basic Shaftless | Socks | White | bought 3464 times
    0.982 | 7p Basic Shaftless           | Socks            | Black            | same product_code: True
    0.889 | Greta Thong Mynta Low 3p     | Underwear bottom | Black            | same product_code: False
    0.856 | Strap Top 2 pack             | Vest top         | Black            | same produc

**What the network learned to treat as similar:** for articles with plenty of purchases, the nearest neighbours are **colour variants of the same product** (prediction confirmed: 4 of 5 neighbours of the RICHIE HOOD hoodie share its product_code), followed by the same product type (skinny jeans → other jeans, dress → dress / playsuit). Some neighbours can't be explained by the features alone and look like **purchase patterns**: white basic socks sit next to black thongs and a black strap-top 2-pack, the "basics bought together" basket. The rarely-bought dress's neighbours are unrelated articles (a blouse, leggings, a tee) with very high similarity (0.94-0.96): an article with one purchase barely moves its ID embedding, so rare articles end up close together in one "not much known" region rather than near similar products.

## Step 8: What does serving the final system cost?
**Predicted memory (arithmetic first):**
- ALS customer vectors: 467,944 x 64 x 4 bytes = 119,809,664 bytes ≈ **114 MB**
- ALS article vectors: 39,829 x 64 x 4 bytes = 10,196,224 bytes ≈ **9.7 MB**
- Repurchase lists: 467,943 x 10 x 4 bytes = 18,717,720 bytes ≈ **17.9 MB**

**The design question:** the final system only uses purchase history, last week's sales and age bucket. **Nothing depends on the moment of the request** (no session, no time of day, no item just viewed), so it can be served two ways:
- **Real-time:** per request, score all 39,829 articles with the customer's ALS vector, then fill after their repurchases.
- **Precomputed (batch):** once a week, compute every customer's top 10 into one table; the API just looks up a row.

In [20]:
print('ALS customer vectors:', round(als_w10.user_factors.nbytes / 1024**2, 1), 'MB | predicted 114 MB')
print('ALS article vectors: ', round(als_w10.item_factors.nbytes / 1024**2, 1), 'MB | predicted 9.7 MB')
own_array = np.zeros((len(own_lists_w10), 10), dtype='int32')       # the repurchase lists as one int32 table
print('Repurchase lists:    ', round(own_array.nbytes / 1024**2, 1), 'MB | predicted 17.9 MB')

ALS customer vectors: 114.2 MB | predicted 114 MB
ALS article vectors:  9.7 MB | predicted 9.7 MB
Repurchase lists:     17.9 MB | predicted 17.9 MB


In [21]:
import time

# REAL-TIME: one warm customer, everything computed at request time
column_to_article_w10 = column_to_article_array(als_article_map_w10)

def realtime_recommend(customer):
    scores = als_w10.user_factors[als_customer_map_w10[customer]] @ als_w10.item_factors.T     # 39,829 scores
    top30 = column_to_article_w10[np.argpartition(-scores, 30)[:30]]
    top30 = top30[np.argsort(-scores[np.argpartition(-scores, 30)[:30]])]
    return fill_up(own_lists_w10[customer], top30)

example_customer = test_ids[0] if test_ids[0] in als_customer_map_w10 else next(c for c in test_ids if c in als_customer_map_w10)
repeats = 200
start = time.perf_counter()
for _ in range(repeats):
    realtime_recommend(example_customer)
realtime_ms = (time.perf_counter() - start) / repeats * 1000
print('Real-time: one customer takes', round(realtime_ms, 3), 'ms')

Real-time: one customer takes 3.107 ms


In [22]:
# PRECOMPUTED: the weekly batch job, every known customer's final top 10 in one table
all_known = sorted(als_customer_map_w10.keys())
start = time.perf_counter()
batch_lists = final_system_recommend(all_known, history_w10, 10, new_als_parts)
batch_minutes = (time.perf_counter() - start) / 60

batch_table = np.array(batch_lists, dtype='int32')                        # (467,943, 10)
batch_row = {customer: row for row, customer in enumerate(all_known)}
cold_lists = {bucket: fill_up(bucket_lists_w10.get(bucket, []), global_top10_w10) for bucket in range(0, 8)}
print('Weekly batch job:', round(batch_minutes, 1), 'minutes for', len(all_known), 'customers')
print('Precomputed table:', batch_table.shape, '=', round(batch_table.nbytes / 1024**2, 1), 'MB (+ 8 cold lists)')

start = time.perf_counter()
for _ in range(100000):
    batch_table[batch_row[example_customer]]
lookup_us = (time.perf_counter() - start) / 100000 * 1e6
print('Precomputed: one lookup takes', round(lookup_us, 2), 'microseconds')
print('Same answer as real-time?', list(batch_table[batch_row[example_customer]]) == [int(a) for a in realtime_recommend(example_customer)])

Weekly batch job: 4.3 minutes for 467943 customers
Precomputed table: (467943, 10) = 17.9 MB (+ 8 cold lists)
Precomputed: one lookup takes 0.35 microseconds
Same answer as real-time? True


**Serving sentence:** I'd serve the **precomputed (batch)** version: nothing in the final system depends on the moment of the request, and a weekly 4.3-minute job produces a 17.9 MB table that answers a request in **0.35 microseconds**, against 3.1 ms and 124 MB of ALS vectors in memory for real-time scoring, with an identical answer (checked). Real-time scoring only earns its cost when the model needs request-time information (a session, an item just viewed), which is where a two-tower model with context features could earn its place in a later version.

## Step 9: The verdict

**Champion: the final system**: repurchase first, then ALS fills the remaining slots; cold customers get last-week popularity in their age bucket.

| Final system | Recall@10 | NDCG@10 | Validation recall |
|---|---|---|---|
| All (68,984) | **0.0507** | **0.0370** | 0.0495 |
| Warm (47,374) | 0.0625 | 0.0467 | 0.0631 |
| Cold (21,610) | 0.0250 | 0.0157 | 0.0216 |

That's **2.3x last-week popularity** (0.0222) on test, and the top-3 ranking (final system > repurchase + fill > ALS + fallback) **held from validation to test**.

**What's served:** option **[A / B]**, as a **precomputed weekly batch** table (17.9 MB, 0.35 µs per lookup), not real-time scoring.

**Not served, and why:**
- **Two-tower Day 10 (ID dropout):** lost to ALS as the warm filler and to popularity for cold customers on validation (met neither decision rule); 0.0167 on test, below last-week popularity.
- **Two-tower Day 9:** cold-start collapse (row 0 never trained); 0.0119 on test.
- **ALS alone:** 0.0343 on test; it can't serve cold customers, and it's beaten by putting repurchases first.
- **Global and last-week popularity:** 0.0122 / 0.0222, no personalisation, 0.03% catalogue coverage.

**Limitations:**
- The two-tower models were frozen at week 9 (not retrained for the test week) and capped at 8 epochs (Day 10) while still improving, so their test numbers understate them.
- Cold articles can't be recommended: articles first sold in week 11 exist in no list.
- One test week only; that week was quieter than validation, and every refreshed model scored higher on it.
- "The customer ID dominates the network's training" is a hypothesis, not measured; the test would be a user tower with no ID at all (future work).

## Day 11 interview question
*"Your project was meant to showcase a neural network, but you didn't ship it. Defend that decision to a hiring manager who expected a deep-learning project."*

> The goal was never "ship a neural network", it was "recommend what people will buy", and I decided up front that the network had to earn its place against strong baselines. I built a full two-tower model in Keras: in-batch softmax with temperature, masking for accidental hits, logQ correction for popularity bias, early stopping on validation recall. I also diagnosed and fixed its cold-start collapse with ID dropout. But on held-out data, a simpler system won: a customer's own recent purchases, topped up with matrix factorisation, and age-group bestsellers for new customers. It scored 0.051 recall@10 on the test week, 2.3 times popularity, and the model ranking held from validation to test, so the decision wasn't luck. Shipping the network anyway would have meant a bigger, slower service that recommends worse, and I'd rather show I can tell the difference. The network is in the repo as a documented challenger with its full diagnosis, and I know where it would earn its place: request-time context, like the session or the item just viewed, which the precomputed system can't use.